# MS MARCO RARS-v12 Anchored Cutoff-Aware RPQ Development

## TL;DR

V11 established a strong rank-64 RPQ 16-byte residual sidecar. V12 tests one
conservative improvement: a single closed-form, cutoff-weighted centroid
update anchored to that unsupervised codebook. The PCA basis, frozen M32
index, 16-byte payload, alpha `0.75`, Top-B `40`, and product partition do not
change.

This notebook first freezes 2,500 genuinely disjoint MS MARCO **training**
queries whose positives occur in the frozen 1M corpus. It then runs five-fold
OOF evaluation for three fixed seeds and materializes a real 1M × 16-byte
sidecar. Run once in a fresh T4 runtime. Do not edit, rerun, or reuse partial
outputs after metrics appear.


## Evidence and safety boundary

The official MS MARCO train query/qrels sources are used only for a
corpus-restricted development task. Every historical 6,980 dev qid is
excluded before encoding. Query selection and fold assignment are fixed by
SHA-256 rules before candidate retrieval.

Passing all gates authorizes only a new independent-confirmation protocol. It
does not open an old RARS holdout, establish official MS MARCO performance,
implement ScaNN, or demonstrate production latency.


In [1]:
import hashlib, json, os, shutil, subprocess, sys, tarfile, urllib.request
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/ravan-chuang/Embedding_Compression_for_RAG_Retrieval.git'
SOURCE_BRANCH = 'codex/rars-v8-cutoff-sidecar'
V12_REPO = Path('/content/Embedding_Compression_for_RAG_Retrieval_rars_v12')
ENV_ROOT = Path('/content/rars-v12-env')
if ENV_ROOT.exists():
    shutil.rmtree(ENV_ROOT)
venv = subprocess.run([
    sys.executable, '-m', 'venv', '--without-pip', '--system-site-packages',
    str(ENV_ROOT),
], text=True, capture_output=True)
if venv.returncode != 0:
    print(venv.stdout)
    print(venv.stderr, file=sys.stderr)
    venv.check_returncode()
EXPERIMENT_PYTHON = str(ENV_ROOT / 'bin/python')
subprocess.run([
    sys.executable, '-m', 'pip', '--python', EXPERIMENT_PYTHON, 'install', '-q',
    'numpy==1.26.4', 'faiss-gpu-cu12==1.12.0', 'pytest>=8,<9',
    'sentence-transformers==3.4.1',
], check=True)
EXPERIMENT_ENV = os.environ.copy()
EXPERIMENT_ENV['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
versions = subprocess.check_output([
    EXPERIMENT_PYTHON, '-c',
    'import importlib.metadata, numpy, torch; '
    'print(numpy.__version__); print(torch.__version__); print(torch.version.cuda); '
    'print(importlib.metadata.version("sentence-transformers"))',
], text=True, env=EXPERIMENT_ENV).splitlines()
assert versions == ['1.26.4', '2.11.0+cu128', '12.8', '3.4.1'], versions

def clone_exact(destination, commit):
    if destination.exists():
        shutil.rmtree(destination)
    subprocess.run(['git', 'clone', '--no-checkout', REPO_URL, str(destination)], check=True)
    subprocess.run(['git', '-C', str(destination), 'checkout', '--detach', commit], check=True)
    actual = subprocess.check_output(['git', '-C', str(destination), 'rev-parse', 'HEAD'], text=True).strip()
    dirty = subprocess.check_output(['git', '-C', str(destination), 'status', '--porcelain'], text=True).strip()
    assert actual == commit and not dirty, (actual, commit, dirty)

resolved = subprocess.check_output([
    'git', 'ls-remote', REPO_URL, f'refs/heads/{SOURCE_BRANCH}'
], text=True).split()[0]
clone_exact(V12_REPO, resolved)
V12_IMPLEMENTATION_COMMIT = resolved

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

SOURCE_HASHES = {
    "protocols/rars_v12_anchored_cutoff_rpq_v1.json": "16940ee2bd5580096ff61819bdb36aadc794a08f928cd92dffa1db11cedae31b",
    "scripts/rars_v12_ca_rpq_core.py": "9e12448c4cdc4d67ce1a9197e8afea2ee6a118181774e27ee1baf178e20136ab",
    "scripts/freeze_rars_v12_fresh_queries.py": "938eaf01ff25cfb3351f0eb973f0050900be09f76c1ef48c4aca94c9d45684cf",
    "scripts/build_rars_v12_fresh_bundle.py": "9a70dec8bed6f37da7aed13df13aab6ca696ac44e76aca2124032492fe16cf66",
    "scripts/train_rars_v12_ca_rpq.py": "9b4a9d5b21c23b7d59aa8688b038d5c05cd60fdfbdbcf238d3cffe29323b9e12",
    "scripts/verify_rars_v12_ca_rpq_packet.py": "429b6880552bf9230eee2c1055a46362a583ffa1b1ce8cf43eb530cef346c1bc",
    "scripts/rars_v11_rank_rate_core.py": "6b7f18226e7d3602cf982794b851b237f17efaee7d83621b4b690b77ec8f0898",
    "scripts/rars_v8_cutoff_sidecar_core.py": "990941bd045461c62241dcc44f5ac450be53ed9c584e88ff6c79e5a9fe845118",
    "scripts/train_rars_v8_cutoff_sidecar.py": "0aba29c636981f740c4bdda89fa0626dd7df17179326b9d0ef36e67226bef946",
    "scripts/evaluate_rars_v6_1m_headroom.py": "1ee10e56b6dd8fbc34ad9976b2e41575e568b3c333a70671abbb43215de22b32"
}
for relative, expected in SOURCE_HASHES.items():
    actual = sha256_file(V12_REPO / relative)
    assert actual == expected, (relative, actual, expected)
print('Exact V12 run commit:', V12_IMPLEMENTATION_COMMIT)
print('All V12 source hashes verified:', versions)


Mounted at /content/drive
Exact V12 run commit: 07b0fe09b82babb3b06ffd1649266a656dd07df1
All V12 source hashes verified: ['1.26.4', '2.11.0+cu128', '12.8', '3.4.1']


In [2]:
PROTOCOL = V12_REPO / 'protocols/rars_v12_anchored_cutoff_rpq_v1.json'
protocol = json.loads(PROTOCOL.read_text())
assert protocol['status'] == 'FROZEN_BEFORE_FIRST_V12_FRESH_DEVELOPMENT_RUN'
assert protocol['fresh_query_freeze']['target_query_count'] == 2500
assert protocol['method']['rank'] == 64
assert protocol['method']['payload_bytes_per_document'] == 16
assert protocol['centroid_update']['updates'] == 1
assert protocol['rpq_training']['all_seeds_run_in_all_folds'] is True
assert protocol['development_gate']['old_holdout_reuse_authorized'] is False
subprocess.run([
    EXPERIMENT_PYTHON, '-m', 'pytest', '-q',
    'tests/test_rars_v12_ca_rpq_core.py',
    'tests/test_rars_v12_protocol_contract.py',
    'tests/test_rars_v12_pipeline_contract.py',
    'tests/test_rars_v11_rank_rate_core.py',
    'tests/test_rars_v8_cutoff_sidecar_core.py',
], cwd=V12_REPO, env=EXPERIMENT_ENV, check=True)
print('V12 numerical, data-isolation, payload, and decision contracts passed.')


V12 numerical, data-isolation, payload, and decision contracts passed.


## Frozen corpus and official fresh-query sources

The existing document embeddings and M32 IVF-PQ index stay unchanged. Only
the small official query/qrels files are downloaded. The freezer records their
exact byte counts and SHA-256 hashes before selecting or encoding queries.


In [3]:
DRIVE = Path('/content/drive/MyDrive/rag-pq-checkpoints')
CACHE = DRIVE / 'msmarco_basis_gate0_cache'
INDEX = DRIVE / 'msmarco_1m_pq_residual_gate3/frozen_ivfpq_m32_nlist512.index'
V12_ROOT = DRIVE / 'rars-v12-ca-rpq' / V12_IMPLEMENTATION_COMMIT[:12]
QUERY_FREEZE = V12_ROOT / 'fresh-query-freeze'
BUNDLE = V12_ROOT / 'fresh-development-bundle'
DEVELOPMENT = V12_ROOT / 'development-once'
RUNNER_LOGS = V12_ROOT / 'runner-logs'
RUNNER_LOGS.mkdir(parents=True, exist_ok=True)
required = [
    CACHE / 'embeddings.fp16.memmap',
    CACHE / 'doc_ids.int64.memmap',
    INDEX,
]
missing = [str(path) for path in required if not path.is_file()]
assert not missing, {'missing_artifacts': missing}
for path in (QUERY_FREEZE, BUNDLE, DEVELOPMENT):
    assert not path.exists() or not any(path.iterdir()), (
        f'{path} is non-empty. Do not overwrite or reuse a partial V12 run.'
    )
assert shutil.disk_usage('/content').free >= 8_000_000_000, 'Need 8 GB local disk'
assert INDEX.stat().st_size == protocol['frozen_index_contract']['index_bytes']
assert sha256_file(INDEX) == protocol['frozen_index_contract']['index_sha256']
print('Frozen 1M corpus/index verified and V12 outputs are empty.')


Frozen 1M corpus/index verified and V12 outputs are empty.


In [4]:
SOURCE_DATA = Path('/content/rars-v12-msmarco-train-sources')
if SOURCE_DATA.exists():
    shutil.rmtree(SOURCE_DATA)
SOURCE_DATA.mkdir(parents=True)
queries_archive = SOURCE_DATA / 'queries.tar.gz'
qrels_train = SOURCE_DATA / 'qrels.train.tsv'
queries_train = SOURCE_DATA / 'queries.train.tsv'
urllib.request.urlretrieve(
    protocol['fresh_query_freeze']['official_queries_archive_url'],
    queries_archive,
)
urllib.request.urlretrieve(
    protocol['fresh_query_freeze']['official_qrels_url'],
    qrels_train,
)
with tarfile.open(queries_archive, 'r:gz') as archive:
    members = [
        member for member in archive.getmembers()
        if Path(member.name).name == protocol['fresh_query_freeze']['queries_member']
    ]
    assert len(members) == 1, [member.name for member in members]
    source = archive.extractfile(members[0])
    assert source is not None
    queries_train.write_bytes(source.read())
assert sum(1 for _ in queries_train.open()) > 500000
assert sum(1 for _ in qrels_train.open()) > 500000
print(json.dumps({
    'queries_train_bytes': queries_train.stat().st_size,
    'queries_train_sha256': sha256_file(queries_train),
    'qrels_train_bytes': qrels_train.stat().st_size,
    'qrels_train_sha256': sha256_file(qrels_train),
}, indent=2))


{
  "queries_train_bytes": 35157438,
  "queries_train_sha256": "4583ebd42180bf0de26a06024d4679fdf627f33cce4b0048ae4c6fc85254506a",
  "qrels_train_bytes": 10589532,
  "qrels_train_sha256": "641b3c9391ea19e4a3d9e3284299f07be6725ff2dd11591a3d4d3f293db17cf2"
}


## Pre-candidate fresh-query freeze

This is the only query-selection step. It excludes all 6,980 historical dev
qids, applies the fixed corpus-coverage and SHA-256 rule, encodes exactly 2,500
queries with the pinned MiniLM revision, and freezes five folds. No search or
metric is performed here.


In [5]:
QUERY_FREEZE.mkdir(parents=True, exist_ok=True)
freezer = subprocess.run([
    EXPERIMENT_PYTHON, str(V12_REPO / 'scripts/freeze_rars_v12_fresh_queries.py'),
    '--queries-train', str(queries_train),
    '--qrels-train', str(qrels_train),
    '--doc-ids', str(CACHE / 'doc_ids.int64.memmap'),
    '--prior-qids', str(V12_REPO / 'splits/msmarco_rars_train_qids.json'),
    '--prior-qids', str(V12_REPO / 'splits/msmarco_rars_validation_qids.json'),
    '--prior-qids', str(V12_REPO / 'splits/msmarco_rars_test_qids.json'),
    '--output-dir', str(QUERY_FREEZE),
    '--protocol', str(PROTOCOL),
    '--source-commit', V12_IMPLEMENTATION_COMMIT,
    '--target-query-count', '2500',
    '--device', 'cuda',
], text=True, capture_output=True, cwd=V12_REPO, env=EXPERIMENT_ENV)
(RUNNER_LOGS / 'query_freeze_stdout.log').write_text(freezer.stdout)
(RUNNER_LOGS / 'query_freeze_stderr.log').write_text(freezer.stderr)
if freezer.returncode != 0:
    print(freezer.stdout[-12000:])
    print(freezer.stderr[-12000:])
    freezer.check_returncode()
query_freeze = json.loads((QUERY_FREEZE / 'fresh_query_freeze.json').read_text())
query_manifest = json.loads((QUERY_FREEZE / 'fresh_query_manifest.json').read_text())
assert query_freeze['status'] == 'RARS_V12_FRESH_QUERY_FREEZE_COMPLETE'
assert query_freeze['selection']['candidate_retrieval_performed'] is False
assert query_manifest['query_count'] == 2500
assert query_manifest['historical_qid_overlap'] == []
assert min(query_manifest['fold_counts']) >= 400
print(json.dumps({
    'selected_qid_hash': query_freeze['selection']['selected_qid_hash'],
    'fold_counts': query_manifest['fold_counts'],
    'positive_qrels_in_frozen_corpus': query_manifest['positive_qrels_in_frozen_corpus'],
}, indent=2))


{
  "selected_qid_hash": "2456fac410f941ee7de24a812d9a9f50183b35daea2892ebf26fb226fbb950e0",
  "fold_counts": [
    498,
    451,
    506,
    535,
    510
  ],
  "positive_qrels_in_frozen_corpus": 2521
}


## Candidate/residual bundle

Only after the query freeze is durable, retrieve the frozen M32 Top-100,
materialize labels for the corpus-restricted qrels, and reconstruct residuals
for the candidate union. This stage still computes no ranking metric.


In [6]:
BUNDLE.mkdir(parents=True, exist_ok=True)
bundler = subprocess.run([
    EXPERIMENT_PYTHON, str(V12_REPO / 'scripts/build_rars_v12_fresh_bundle.py'),
    '--query-freeze-root', str(QUERY_FREEZE),
    '--embeddings', str(CACHE / 'embeddings.fp16.memmap'),
    '--doc-ids', str(CACHE / 'doc_ids.int64.memmap'),
    '--index', str(INDEX),
    '--output-dir', str(BUNDLE),
    '--protocol', str(PROTOCOL),
    '--source-commit', V12_IMPLEMENTATION_COMMIT,
], text=True, capture_output=True, cwd=V12_REPO, env=EXPERIMENT_ENV)
(RUNNER_LOGS / 'bundle_stdout.log').write_text(bundler.stdout)
(RUNNER_LOGS / 'bundle_stderr.log').write_text(bundler.stderr)
if bundler.returncode != 0:
    print(bundler.stdout[-12000:])
    print(bundler.stderr[-12000:])
    bundler.check_returncode()
bundle_manifest = json.loads((BUNDLE / 'fresh_bundle_manifest.json').read_text())
assert bundle_manifest['status'] == 'RARS_V12_FRESH_DEVELOPMENT_BUNDLE_FROZEN'
assert bundle_manifest['metrics_computed'] is False
assert bundle_manifest['old_rars_holdout_opened'] is False
print(json.dumps({
    'query_count': bundle_manifest['query_count'],
    'candidate_residual_count': bundle_manifest['candidate_residual_count'],
    'positive_candidate_hits': bundle_manifest['positive_candidate_hits'],
    'fold_counts': bundle_manifest['fold_counts'],
}, indent=2))


{
  "query_count": 2500,
  "candidate_residual_count": 175790,
  "positive_candidate_hits": 882,
  "fold_counts": [
    498,
    451,
    506,
    535,
    510
  ]
}


## Five-fold, three-seed V12 development

Every fold refits PCA64 and the unsupervised RPQ initializer using the other
four folds. The challenger receives exactly one anchored closed-form centroid
update. After all OOF arrays are complete, the export-only all-development
model writes a real 16,000,000-byte full-corpus sidecar. This cell can take
tens of minutes; do not interrupt it.


In [7]:
DEVELOPMENT.mkdir(parents=True, exist_ok=True)
trainer = subprocess.run([
    EXPERIMENT_PYTHON, str(V12_REPO / 'scripts/train_rars_v12_ca_rpq.py'),
    '--bundle-root', str(BUNDLE),
    '--embeddings', str(CACHE / 'embeddings.fp16.memmap'),
    '--index', str(INDEX),
    '--output-dir', str(DEVELOPMENT),
    '--protocol', str(PROTOCOL),
    '--source-commit', V12_IMPLEMENTATION_COMMIT,
    '--full-corpus-batch-size', '10000',
], text=True, capture_output=True, cwd=V12_REPO, env=EXPERIMENT_ENV)
(RUNNER_LOGS / 'development_stdout.log').write_text(trainer.stdout)
(RUNNER_LOGS / 'development_stderr.log').write_text(trainer.stderr)
if trainer.returncode != 0:
    print('V12 trainer return code:', trainer.returncode)
    print('===== STDOUT =====')
    print(trainer.stdout[-16000:])
    print('===== STDERR =====')
    print(trainer.stderr[-16000:])
    trainer.check_returncode()
print('V12 fresh-query OOF development and full-corpus sidecar completed.')


V12 fresh-query OOF development and full-corpus sidecar completed.


In [8]:
verification = json.loads(subprocess.check_output([
    EXPERIMENT_PYTHON, str(V12_REPO / 'scripts/verify_rars_v12_ca_rpq_packet.py'),
    '--packet-root', str(DEVELOPMENT),
    '--repo-root', str(V12_REPO),
], text=True, cwd=V12_REPO, env=EXPERIMENT_ENV))
result = json.loads((DEVELOPMENT / 'development_result.json').read_text())
complete = json.loads((DEVELOPMENT / 'development_complete.json').read_text())
assert verification['status'] == 'RARS_V12_PACKET_VERIFIED'
assert verification['formal_decision'] == result['formal_decision'] == complete['formal_decision']
assert result['v9_packet_opened'] is False
assert result['v10_packet_opened'] is False
assert result['v11_packet_opened'] is False
assert result['old_holdout_opened'] is False
assert result['fresh_confirmation_access_authorized'] is False
assert (DEVELOPMENT / 'full_corpus_ca_rpq_codes.uint8.memmap').stat().st_size == 16000000
print(json.dumps({
    'formal_decision': result['formal_decision'],
    'metrics': result['metrics'],
    'comparisons': result['comparisons'],
    'seed_gains': result['seed_gains'],
    'fold_gains': result['fold_gains'],
    'candidate_gap_recovery_fraction': result['candidate_gap_recovery_fraction'],
    'maximum_centroid_drift_fraction': result['maximum_centroid_drift_fraction'],
    'failed_gates': result['decision']['failed_gates'],
    'packet_verification': verification,
}, indent=2, allow_nan=False))


{
  "formal_decision": "STOP_CA_RPQ_NO_STABLE_ADVANTAGE",
  "metrics": {
    "base": {
      "recall": 0.1296,
      "mrr": 0.05449253968253968,
      "ndcg": 0.07187174516450145
    },
    "same_candidate_exact": {
      "recall": 0.2374,
      "mrr": 0.12397238095238095,
      "ndcg": 0.15075446514747573
    },
    "unsupervised_primary": {
      "recall": 0.1872,
      "mrr": 0.08395587301587301,
      "ndcg": 0.10823691802291706
    },
    "ca_rpq_primary": {
      "recall": 0.1874,
      "mrr": 0.0846515873015873,
      "ndcg": 0.10881980485756182
    },
    "unsupervised_by_seed": [
      {
        "recall": 0.1872,
        "mrr": 0.08395587301587301,
        "ndcg": 0.10823691802291706
      },
      {
        "recall": 0.1854,
        "mrr": 0.08495126984126984,
        "ndcg": 0.10854603089351719
      },
      {
        "recall": 0.1856,
        "mrr": 0.0867515873015873,
        "ndcg": 0.10986246825491619
      }
    ],
    "ca_rpq_by_seed": [
      {
        "recall": 0.18

## Handoff

Accept the printed decision unchanged. Copy the completed development packet
and the small lineage manifests into a download archive. Return that ZIP and
this executed notebook for independent audit. Do not rerun after inspecting
the result.


In [9]:
PACKET = Path('/content') / f'rars-v12-ca-rpq-{V12_IMPLEMENTATION_COMMIT[:12]}'
if PACKET.exists():
    shutil.rmtree(PACKET)
(PACKET / 'development').mkdir(parents=True)
(PACKET / 'lineage').mkdir(parents=True)
for path in DEVELOPMENT.iterdir():
    if path.is_file():
        shutil.copy2(path, PACKET / 'development' / path.name)
for path in (
    QUERY_FREEZE / 'fresh_query_freeze.json',
    QUERY_FREEZE / 'fresh_query_manifest.json',
    QUERY_FREEZE / 'fresh_qrels.json',
    BUNDLE / 'fresh_bundle_manifest.json',
    BUNDLE / 'fresh_bundle_complete.json',
):
    shutil.copy2(path, PACKET / 'lineage' / path.name)
archive = Path(shutil.make_archive(str(PACKET), 'zip', root_dir=PACKET))
print('Download:', archive, archive.stat().st_size, 'bytes')
from google.colab import files
files.download(str(archive))


Download: /content/rars-v12-ca-rpq-07b0fe09b82b.zip 16321391 bytes


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>